# 08 — CSE-CIC-IDS2018 matched join (Tuesday 20-02-2018 only)

The one day whose official CSV retains Flow ID / IPs / ports. Reuses the
exact machinery of notebook 03: per-element parsing, 12-h repair if the
signature appears, shift search ±12 h, canonical endpoint key, minute
resolution, ambiguous-key drop, transition matrix.

Gate G2 (≥20% match) applies unchanged.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
DRIVE_ROOT = '/content/drive/MyDrive/research/ids-label-correction'
sys.path.insert(0, os.path.join(DRIVE_ROOT, 'src'))

import importlib, config as C, helpers as H
importlib.reload(C); importlib.reload(H)
import pandas as pd, numpy as np, json

RAW18 = os.path.join(DRIVE_ROOT, 'data/raw_original_2018')
orig_p = [f for f in os.listdir(RAW18) if '20-02-2018' in f]
assert orig_p, 'Thuesday-20-02-2018 file not found in raw_original_2018'
print('original day file:', orig_p[0])

imp_p = None
for dirpath, _, files in os.walk(C.RAW_IMPROVED):
    for fn in files:
        if '20-02-2018' in fn and fn.lower().endswith('.csv'):
            imp_p = os.path.join(dirpath, fn)
print('improved day file:', imp_p)
assert imp_p, 'improved Tuesday-20-02-2018 not found'

Mounted at /content/drive
original day file: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
improved day file: /content/drive/MyDrive/research/ids-label-correction/data/raw_improved/CSECICIDS2018_improved/Tuesday-20-02-2018.csv


In [ ]:
orig = pd.read_csv(os.path.join(RAW18, orig_p[0]), low_memory=False,
                   encoding='latin-1')
orig = H.harmonise(orig)
orig = orig[orig['label'].notna() & (orig['label'] != 'Label')]
impr = H.harmonise(pd.read_csv(imp_p, low_memory=False))
print(f'original {len(orig):,}   improved {len(impr):,}')
missing = [k for k in ['src_ip','src_port','dst_ip','dst_port','protocol','timestamp']
           if k not in orig.columns]
assert not missing, f'original day file lacks {missing}'
print('key columns present on both sides')

original 7,948,748   improved 6,054,702
key columns present on both sides


In [ ]:
# Parse (capture date 20 Feb 2018 => month must be 2), clock repair, keys
def choose_parse(series, name):
    a = pd.to_datetime(series, errors='coerce', dayfirst=True, format='mixed')
    va = float((a.dt.month == 2).mean())
    if va < 0.999:
        b = pd.to_datetime(series, errors='coerce', dayfirst=False, format='mixed')
        vb = float((b.dt.month == 2).mean())
        a, va = (b, vb) if vb > va else (a, va)
    print(f'{name}: month==2 {va:.1%}  NaT {a.isna().mean():.2%}')
    assert va > 0.99, name + ' parse failed — inspect formats'
    return a

ts_o = choose_parse(orig['timestamp'], 'original')
ts_i = choose_parse(impr['timestamp'], 'improved')

for nm in ['o', 'i']:
    ts = ts_o if nm == 'o' else ts_i
    h = ts.dt.hour
    if h.between(1, 7).mean() > 0.02 and h.between(13, 17).mean() < 0.005:
        ts = ts + pd.to_timedelta(h.between(1, 7).astype('int64') * 12, unit='h')
        print('12h repair applied to', 'original' if nm == 'o' else 'improved')
        if nm == 'o': ts_o = ts
        else: ts_i = ts

def keyframe(df, ts, shift_h=0):
    ip_s = df['src_ip'].astype(str).str.strip()
    ip_d = df['dst_ip'].astype(str).str.strip()
    po_s = pd.to_numeric(df['src_port'], errors='coerce').astype('Int64').astype(str)
    po_d = pd.to_numeric(df['dst_port'], errors='coerce').astype('Int64').astype(str)
    ep_a = (ip_s + ':' + po_s).values
    ep_b = (ip_d + ':' + po_d).values
    first = ep_a <= ep_b
    k = pd.DataFrame({'k1': np.where(first, ep_a, ep_b),
                      'k2': np.where(first, ep_b, ep_a),
                      'pr': pd.to_numeric(df['protocol'], errors='coerce')
                              .astype('Int64').astype(str)}, index=df.index)
    k['mn'] = (ts + pd.Timedelta(hours=shift_h)).dt.floor('min')
    return k

iset = set(map(tuple, keyframe(impr, ts_i).astype(str).values))
results = {}
for shift in range(-12, 13):
    ko = keyframe(orig, ts_o, shift)
    results[shift] = sum(1 for t in map(tuple, ko.astype(str).values) if t in iset)
    print(f'shift {shift:+3d}h  {results[shift]:>10,}')
best = max(results, key=results.get)
print(f'\nbest {best:+d}h = {results[best]:,} ({results[best]/len(orig):.1%})')
assert results[best] >= 0.05 * len(orig), 'no viable alignment — inspect keys'
if best: ts_o = ts_o + pd.Timedelta(hours=best)

original: month==2 100.0%  NaT 0.00%
improved: month==2 100.0%  NaT 0.00%
12h repair applied to original
shift -12h           0
shift -11h           0
shift -10h           0
shift  -9h           0
shift  -8h           0
shift  -7h          34
shift  -6h         108
shift  -5h         301
shift  -4h       5,362
shift  -3h      11,383
shift  -2h      17,510
shift  -1h      24,468
shift  +0h      31,522
shift  +1h      40,177
shift  +2h      51,500
shift  +3h      60,357
shift  +4h   7,412,302
shift  +5h      60,401
shift  +6h      51,577
shift  +7h      40,234
shift  +8h      31,564
shift  +9h      24,546
shift +10h      17,589
shift +11h      11,447
shift +12h       5,387

best +4h = 7,412,302 (93.3%)


In [ ]:
KEY = ['k1', 'k2', 'pr', 'mn']
def with_key(df, ts):
    df = df.copy()
    k = keyframe(df, ts)
    for c in KEY: df[c] = k[c]
    return df

def unique_keyed(df, name):
    df = df.dropna(subset=['mn'])
    counts = df.groupby(KEY, dropna=False).size()
    dup = counts[counts > 1].index
    df = df.set_index(KEY)
    df = df[~df.index.isin(dup)].reset_index()
    print(f'{name}: unique-keyed {len(df):,}')
    return df

ou = unique_keyed(with_key(orig, ts_o), 'original')
iu = unique_keyed(with_key(impr, ts_i), 'improved')
extra = ['label'] + (['attempted'] if 'attempted' in iu.columns else [])
matched = ou.merge(iu[KEY + extra], on=KEY, how='inner', suffixes=('', '_improved'))
if 'label_improved' not in matched.columns and 'label_y' in matched.columns:
    matched = matched.rename(columns={'label_y': 'label_improved', 'label_x': 'label'})
matched = matched.rename(columns={'label': 'label_original'})
rate = len(matched) / len(ou)
print(f'matched {len(matched):,}  ({rate:.2%} of unique original)  '
      f'G2 {"PASS" if rate >= C.MIN_MATCH_RATE else "FAIL"}')

mo, mi = H.coarse_class(matched['label_original']), H.coarse_class(matched['label_improved'])
xt = pd.crosstab(mo, mi, rownames=['original'], colnames=['improved'])
xt.to_csv(os.path.join(C.RESULTS, 't18_transition_matrix.csv'))
ch = (matched['label_original'].astype(str).str.strip().str.upper()
      != matched['label_improved'].astype(str).str.strip().str.upper())
bo, bi = H.binarise(matched['label_original']), H.binarise(matched['label_improved'])
summ = {'matched': len(matched), 'changed': int(ch.sum()),
        'change_pct': round(ch.mean()*100, 3),
        'benign_to_attack': int(((bo==0)&(bi==1)).sum()),
        'attack_to_benign': int(((bo==1)&(bi==0)).sum()),
        'match_rate_pct': round(rate*100, 2)}
json.dump(summ, open(os.path.join(C.RESULTS, 't18_match_summary.json'), 'w'), indent=2)
print(json.dumps(summ, indent=2)); xt

original: unique-keyed 4,985,300
improved: unique-keyed 6,000,994
matched 4,575,969  (91.79% of unique original)  G2 PASS
{
  "matched": 4575969,
  "changed": 10116,
  "change_pct": 0.221,
  "benign_to_attack": 3,
  "attack_to_benign": 0,
  "match_rate_pct": 91.79
}


improved,BENIGN,DDoS
original,,
BENIGN,4565853,3
DDoS,0,10113


In [ ]:
drop = [c for c in ['k1','k2','pr','mn'] if c in matched.columns]
matched.drop(columns=drop).to_parquet(
    os.path.join(C.INTERIM, 'matched_2018.parquet'), index=False)
print('cached matched_2018.parquet:', len(matched))

cached matched_2018.parquet: 4575969


In [1]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os, pandas as pd
DRIVE_ROOT = '/content/drive/MyDrive/research/ids-label-correction'
sys.path.insert(0, os.path.join(DRIVE_ROOT, 'src'))
import config as C

m = pd.read_parquet(os.path.join(C.INTERIM, 'matched_2018.parquet'))
ch = m['label_original'].str.strip().str.upper() != m['label_improved'].str.strip().str.upper()
print(m.loc[ch, ['label_original', 'label_improved']].value_counts().head(10))

Mounted at /content/drive
label_original          label_improved
DDoS attacks-LOIC-HTTP  DDoS-LOIC-HTTP    9316
                        DDoS-LOIC-UDP      797
Benign                  DDoS-LOIC-HTTP       3
Name: count, dtype: int64


In [2]:
dec = m.loc[ch, ['label_original', 'label_improved']].value_counts().reset_index(name='n')
dec.to_csv(os.path.join(C.RESULTS, 't18_changed_decomposition.csv'), index=False)
print('saved', len(dec), 'rows')

saved 3 rows
